# Stage 02: Image Preprocessing

**Status:** Implemented, fully dataset-agnostic. Deterministic RGB → Gamma Correction → CLAHE →
Processed RGB PNG -- no other transform. See `PROJECT_CODE.md`'s "Stage 02 Preprocessing Policy"
and `PROJECT_STRUCTURE.md`'s Pipeline Overview for the specification this notebook implements;
`SEGMENTATION_ARCHITECTURE.md` for why RGB (not single-channel) and no resize.

## Objective

Apply `image_preprocessing.py`'s approved Stage 02 pipeline (Gamma Correction, then CLAHE; no
Green Channel Extraction, Ben Graham, Median Filtering, Histogram Equalization, resizing, or
augmentation) to every fundus image in every approved dataset under `datasets/`, writing the
result once to each dataset's `processed/` folder. This notebook does not train anything -- it is
a deterministic transform, not a trained model.

## Dataset discovery -- this notebook contains no dataset names

Earlier versions of this notebook hardcoded `APTOS2019` and `IDRiD`. This version discovers what
to preprocess at run time instead, so adding a new dataset (DRIVE, CHASE_DB1, or anything added
later) never requires editing this notebook:

1. Recursively search under `datasets/` (on Google Drive, where the real data lives) for every
   directory that has an immediate `raw/` child -- each one is a preprocessing target (a dataset,
   or a dataset's subtask, e.g. `IDRiD/grading` and `IDRiD/localization` are each their own target
   since each has its own `raw/`).
2. **Skip `EyeQ` completely** -- the one, explicitly-named exception (see Section 3). Stage 1
   (Image Quality Assessment) is trained on, and must continue to see, the original unprocessed
   RGB EyeQ images; Stage 02 never touches `datasets/EyeQ/`.
3. Within each remaining target's `raw/` subtree, only leaf folders that actually contain fundus
   images (by file extension) become preprocessing jobs -- a folder of label CSVs (e.g. IDRiD's
   `2. Groundtruths/`) or an empty `raw/` (e.g. IDRiD's `segmentation`, not yet populated) is
   never treated as a job, so masks/CSVs/labels are never preprocessed and nothing crashes on an
   empty dataset.
4. Output preserves the exact folder hierarchy found under `raw/`, mirrored under `processed/` --
   e.g. `IDRiD/grading/raw/1. Original Images/a. Training Set` →
   `IDRiD/grading/processed/1. Original Images/a. Training Set`. `raw/` itself is only ever read,
   never modified.

**Known, accepted consequence of full genericity:** IDRiD's `localization` subset has its own
`raw/` folder and will be discovered and processed alongside `grading`, even though its source
images are byte-for-byte identical to `grading`'s (verified by SHA-256 in an earlier session) --
this notebook does not special-case that, since doing so would reintroduce exactly the kind of
dataset-specific reference this refactor removes. A future consumer of localization-labeled images
can still read from `IDRiD/grading/processed/` by filename instead if avoiding the duplicate is
ever worth a dedicated design change; that is a decision for a future task, not this one.

## Before running

`Runtime > Change runtime type > Hardware accelerator > None` -- this stage is a CPU-bound OpenCV
transform (Gamma LUT + CLAHE), not a trained model; it does not use a GPU.

### Bootstrap

Identical to `stage01_iqa.ipynb`'s Bootstrap cell -- the minimal clone + `sys.path` setup every
stage notebook needs before `colab/common/` is importable.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

for path in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if path not in sys.path:
        sys.path.insert(0, path)

print("Bootstrap complete:", REPO_DIR)

## 1. Setup

`setup.setup()` mounts Google Drive, clones/updates the repository, installs
`requirements.txt`, and enters the repository -- identical call to `stage01_iqa.ipynb`'s Section 1,
reused unmodified.

In [ ]:
import setup

setup_info = setup.setup()

## 2. Environment Verification

`verify_environment.verify_all()` checks Python/TensorFlow versions, the repository path, Google
Drive, and required packages -- same call as Stage 1, except **`require_gpu=False`**: Stage 02 is
a CPU-bound OpenCV transform, not a trained model, so it does not need a GPU runtime.

In [ ]:
import colab_config
import verify_environment

env_report = verify_environment.verify_all(
    repo_dir=colab_config.REPO_DIR,
    drive_mount_point=colab_config.DRIVE_MOUNT_POINT,
    requirements_path=os.path.join(colab_config.REPO_DIR, "requirements.txt"),
    require_gpu=False,
)

## 3. Dataset Staging

Discovers every preprocessing target directly against the Drive-mounted `datasets/` tree (the
only place the real data exists before staging -- a fresh Colab clone never ships dataset
contents, per `PROJECT_CODE.md`'s Training policy), then stages each discovered target's `raw/`
folder onto the local SSD via `dataset_staging.stage_dataset()` (reused unmodified -- already
dataset-agnostic by design, so no changes were needed here) and verifies each copy
(`dataset_staging.verify_staged_copy()`).

**`SKIP_DATASET_NAMES = {"EyeQ"}` is the one dataset-specific reference this notebook contains,
and it is explicitly required** (see Section title in the overview above) -- everything else below
is driven entirely by what `discover_preprocessing_targets()` finds on disk.

In [ ]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
SKIP_DATASET_NAMES = {"EyeQ"}


def discover_preprocessing_targets(datasets_root, skip_names=SKIP_DATASET_NAMES):
    """Recursively finds every directory under `datasets_root` with an immediate
    `raw/` child -- each is a preprocessing target. Skips any target whose
    relative path contains a name in `skip_names`. For each remaining
    target, walks its `raw/` subtree and keeps only leaf folders that
    directly contain fundus images (by extension) -- label/mask/CSV-only
    folders and empty raw/ trees are silently excluded, never treated as a
    job. Returns a list of dicts: dataset_label, raw_dir (Drive path),
    dataset_root (Drive path), and image_subfolders (relative path within
    raw/ -> image count), preserving the exact folder hierarchy found.
    """
    targets = []
    for dirpath, dirnames, filenames in os.walk(datasets_root):
        if "raw" not in dirnames:
            continue
        rel_root = os.path.relpath(dirpath, datasets_root)
        path_parts = rel_root.split(os.sep)
        # A "raw" or "processed" folder never contains a real nested dataset root.
        dirnames[:] = [d for d in dirnames if d not in ("raw", "processed")]
        if any(part in skip_names for part in path_parts):
            continue

        raw_dir = os.path.join(dirpath, "raw")
        image_subfolders = []
        for sub_dirpath, _, sub_filenames in os.walk(raw_dir):
            images = [f for f in sub_filenames if f.lower().endswith(IMAGE_EXTENSIONS)]
            if not images:
                continue
            rel_to_raw = os.path.relpath(sub_dirpath, raw_dir)
            image_subfolders.append({"relative_path": rel_to_raw, "image_count": len(images)})

        if not image_subfolders:
            continue  # has raw/, but no fundus images anywhere in it (e.g. empty, or masks/CSVs only)

        targets.append({
            "dataset_label": rel_root.replace(os.sep, "/"),
            "raw_dir": raw_dir,
            "dataset_root": dirpath,
            "image_subfolders": image_subfolders,
        })
    return targets


discovered_targets = discover_preprocessing_targets(colab_config.DRIVE.datasets_root)

print(f"Discovered {len(discovered_targets)} preprocessing target(s):")
for target in discovered_targets:
    print(f"  {target['dataset_label']}")
    for sf in target["image_subfolders"]:
        rel = sf["relative_path"] if sf["relative_path"] != "." else "(raw/ root)"
        print(f"    {rel}: {sf['image_count']} images")

assert "EyeQ" not in [t["dataset_label"] for t in discovered_targets], "EyeQ must never be a target"


In [ ]:
import shutil

# One-time safeguard against a partial local copy left behind by an interrupted
# previous session -- on a clean VM these have nothing to delete and are a no-op.
for target in discovered_targets:
    safe_name = target["dataset_label"].replace("/", "_")
    local_path = f"/content/datasets/{safe_name}"
    if os.path.exists(local_path):
        shutil.rmtree(local_path)

print("Deleted any incomplete staged datasets.")

In [ ]:
import dataset_staging

# Each target is staged once as a whole (its raw/ root), covering every
# split/subfolder it contains in a single copy. The resulting JOBS list is
# built here too, one entry per image-bearing subfolder discovered above,
# now pointing at the staged local paths.
JOBS = []
for target in discovered_targets:
    safe_name = target["dataset_label"].replace("/", "_")
    staged = dataset_staging.stage_dataset(target["raw_dir"], safe_name)
    dataset_staging.verify_staged_copy(staged)

    for sf in target["image_subfolders"]:
        rel = sf["relative_path"]
        local_raw_dir = staged.local_dir if rel == "." else os.path.join(staged.local_dir, rel)
        processed_suffix = "" if rel == "." else os.sep + rel
        JOBS.append({
            "label": target["dataset_label"] + ("" if rel == "." else "/" + rel.replace(os.sep, "/")),
            "local_raw_dir": local_raw_dir,
            "local_processed_dir": f"/content/processed/{safe_name}{processed_suffix}",
            "drive_processed_dir": os.path.join(target["dataset_root"], "processed")
                                    if rel == "." else os.path.join(target["dataset_root"], "processed", rel),
        })

print(f"\n{len(JOBS)} preprocessing job(s) staged and ready:")
for job in JOBS:
    print(f"  {job['label']}")

## 4. Dataset Verification

Runs `verify_dataset.verify_image_folder()` (reused unmodified -- already dataset-agnostic,
added specifically so this notebook and any future stage don't need EyeQ's `labels.csv`-specific
`verify_eyeq_dataset()`) against every staged raw job folder: directory exists, contains at least
one image, and a random sample decodes without corruption.

In [ ]:
import verify_dataset

raw_reports = {job["label"]: verify_dataset.verify_image_folder(job["local_raw_dir"]) for job in JOBS}

## 5. Preprocessing Configuration

`profile="DR"` (`config.PREPROCESSING_PROFILES.DR`) is Stage 02's approved recipe: Gamma
Correction using `config.PREPROCESSING.DEFAULT_GAMMA`, then CLAHE using
`config.PREPROCESSING.DEFAULT_CLAHE_CLIP_LIMIT` / `DEFAULT_CLAHE_TILE_GRID_SIZE`. Identical for
every job in `JOBS` -- nothing about this stage is dataset-specific. Printed below for this run's
own record, since this notebook has no `experiment_manager`-style `metadata.json` (Stage 02 is
not a training run).

In [ ]:
import config
from image_preprocessing import preprocess_folder

PROFILE = "DR"

print(f"profile: {PROFILE}")
print(f"gamma: {config.PREPROCESSING.DEFAULT_GAMMA}")
print(f"clahe_clip_limit: {config.PREPROCESSING.DEFAULT_CLAHE_CLIP_LIMIT}")
print(f"clahe_tile_grid_size: {config.PREPROCESSING.DEFAULT_CLAHE_TILE_GRID_SIZE}")

## 6. Preprocessing

Reads and writes entirely on the local SSD for every job in `JOBS`, avoiding the per-file Google
Drive FUSE latency `dataset_staging.py`'s own docstring documents -- Section 8 copies the finished
local output back to Drive in bulk afterward. Uses `image_preprocessing.preprocess_folder()`
unmodified for every job, regardless of which dataset it came from. Each job also writes a
`log_file` CSV (already part of `preprocess_folder()`) as its manifest.

In [ ]:
LOG_DIR = "/content/processed/_logs"
os.makedirs(LOG_DIR, exist_ok=True)

preprocessing_results = {}
for job in JOBS:
    log_file = os.path.join(LOG_DIR, job["label"].replace("/", "_") + ".csv")
    result = preprocess_folder(
        job["local_raw_dir"], job["local_processed_dir"], profile=PROFILE, log_file=log_file,
    )
    preprocessing_results[job["label"]] = result
    print(f"[{job['label']}] processed={result.summary.processed} "
          f"skipped={result.summary.skipped} failed={result.summary.failed}")

## 7. Output Verification

Confirms every job produced zero failures, then re-runs `verify_dataset.verify_image_folder()`
against each local processed folder as an independent check that the written files are
themselves valid, decodable images -- not just that `preprocess_folder()` reported success.

In [ ]:
for job in JOBS:
    result = preprocessing_results[job["label"]]
    if result.summary.failed > 0:
        print(f"[WARN] {job['label']}: {result.summary.failed} image(s) failed to preprocess.")
    assert result.summary.total_images == (
        result.summary.processed + result.summary.skipped + result.summary.failed
    ), f"{job['label']}: summary counts do not add up"

processed_reports = {
    job["label"]: verify_dataset.verify_image_folder(job["local_processed_dir"]) for job in JOBS
}

print("\nAll processed output folders verified (see per-folder counts above).")

## 8. Export to Google Drive

Bulk-copies each job's finished local `processed/` folder (plus its manifest CSV) up to the
corresponding Drive-mounted `processed/` destination computed during discovery (Section 3) --
once, after processing is complete. The copy helper mirrors `dataset_staging.py`'s own
`_copy_one` / thread-pool pattern for the reverse direction (that module only copies
Drive-to-local) -- the smallest amount of new glue needed to reuse that same proven approach,
without modifying that module.

In [ ]:
import concurrent.futures
import shutil as _shutil

def _copy_one_to_drive(src, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    _shutil.copy2(src, dst)

def copy_tree_to_drive(local_dir, drive_dir, max_workers=16):
    file_pairs = []
    for dirpath, _, filenames in os.walk(local_dir):
        rel_dir = os.path.relpath(dirpath, local_dir)
        dest_dir = drive_dir if rel_dir == "." else os.path.join(drive_dir, rel_dir)
        for name in filenames:
            file_pairs.append((os.path.join(dirpath, name), os.path.join(dest_dir, name)))

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = [pool.submit(_copy_one_to_drive, src, dst) for src, dst in file_pairs]
        for future in concurrent.futures.as_completed(futures):
            future.result()
    return len(file_pairs)

exported_counts = {}
for job in JOBS:
    count = copy_tree_to_drive(job["local_processed_dir"], job["drive_processed_dir"])
    log_src = os.path.join(LOG_DIR, job["label"].replace("/", "_") + ".csv")
    log_dst = os.path.join(os.path.dirname(job["drive_processed_dir"]), "_logs", os.path.basename(log_src))
    _copy_one_to_drive(log_src, log_dst)
    exported_counts[job["label"]] = count
    print(f"[{job['label']}] exported {count} file(s) -> {job['drive_processed_dir']}")

## 9. Summary

In [ ]:
print("=" * 72)
print("STAGE 02 PREPROCESSING -- RUN SUMMARY")
print("=" * 72)
print(f"Datasets discovered ({len(discovered_targets)}): "
      f"{[t['dataset_label'] for t in discovered_targets]}")
print("EyeQ: skipped (Stage 1 only, never touched by Stage 02).\n")

for job in JOBS:
    result = preprocessing_results[job["label"]]
    print(f"{job['label']}:")
    print(f"  raw images:  {raw_reports[job['label']].image_count}")
    print(f"  processed:   {result.summary.processed}  "
          f"skipped: {result.summary.skipped}  failed: {result.summary.failed}")
    print(f"  exported to: {job['drive_processed_dir']}  ({exported_counts[job['label']]} files)")

print("\nThis notebook required no edits to reach the datasets above, and will require none for")
print("DRIVE, CHASE_DB1, or any future dataset added under datasets/<name>/raw/ -- add the data,")
print("re-run from Section 3.")
print("\nNext step in the pipeline (see PROJECT_CODE.md / IMPLEMENTATION_PLAN.md): Stage 03 --")
print("Vessel Segmentation, once DRIVE and CHASE_DB1 are placed under datasets/.")